# 🍣 FoodVision Mini
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shalev396/ml-lab/blob/main/foodvision-mini/training/notebook.ipynb)

Classify a food photo as **pizza, steak or sushi**. An ImageNet-pretrained EfficientNet-B2 is kept
frozen as a feature extractor and only a new 3-class head is trained, on 450 Food-101 photos.
This notebook drives every step (data, model, training, evaluation against the deployed checkpoint,
inference, export) by calling the functions in `src/`, and writes the Hugging Face model folder
that the [Space](https://huggingface.co/spaces/shalev396/foodvision-mini) serves.

**Contents:** 1. Setup · 2. Config · 3. Data · 4. Load model · 5. Training · 6. Evaluation · 7. Inference · 8. Export

## 1. Setup
1a finds the project files (clones the repo on a fresh runtime), 1b installs the dependencies from the root `requirements.txt`, 1c picks the `device`.

In [ ]:
# 1a. Project + files — fetch the repo / model code only if they aren't here (e.g. a fresh Colab runtime)
import os, subprocess
from pathlib import Path

SLUG = "foodvision-mini"
if not Path("src").is_dir():                      # not inside <slug>/training yet -> get the repo
    if not Path("ml-lab").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/shalev396/ml-lab.git"], check=True)
    os.chdir(f"ml-lab/{SLUG}/training")
if not Path("../model/model.py").exists():   # model/ is a git submodule (HF repo): code only, skip weights
    subprocess.run(["git", "submodule", "update", "--init", "--depth", "1", f"{SLUG}/model"],
                   cwd="../..", check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
print("working dir:", Path.cwd())

In [ ]:
# 1b. Dependencies — the root requirements.txt is the single source of truth for every project.
# Colab/Jupyter never install a repo's requirements on their own, so this cell does it (fast no-op when satisfied).
%pip install -q -r ../../requirements.txt

In [ ]:
# 1c. Device — whatever this machine has (cuda -> mps -> cpu); nothing platform-specific is installed.
import torch
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

## 2. Config
Every hyperparameter lives in `src/config.py`. The defaults are the original recipe: Adam lr 1e-3,
batch 32, 10 epochs, seed 42, dropout 0.3, no augmentation. `SMOKE_TEST=1` shrinks the run to
8 images per class and 1 epoch (a sanity check that takes about a minute) and exports to
`outputs/smoke/model` instead of `../model`.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image as ShowImage, display
from PIL import Image
from torch import nn

from src import data_setup, engine, export, model_builder, utils   # importing src puts ../model on sys.path
from src.config import Config
import model as M                                                    # ../model/model.py, what the Space runs

os.environ.setdefault("SMOKE_TEST", "0")   # "1" = quick smoke run; a SMOKE_TEST set outside the notebook wins
PUSH_TO_HUB = False                         # upload the exported model folder to the Hub (section 8)

cfg = Config()
ASSETS = cfg.run_dir / "assets"            # plots of this run; export copies them into model/assets
print(utils.lib_versions("torch", "torchvision", "huggingface_hub", "numpy"))
print("export to:", cfg.model_dir)
cfg

## 3. Data
`pizza_steak_sushi_20_percent` is a 20% sample of the pizza, steak and sushi classes of
[Food-101](https://huggingface.co/datasets/ethz/food101): 450 train and 150 test images. It is
downloaded once into `training/data/` (GitHub URL, with the raw.githubusercontent.com CDN as
fallback). Train and test images get the same EfficientNet-B2 preprocessing from
`model.get_transform()`: resize 288 (bicubic), center-crop 288, ImageNet mean/std.

In [ ]:
train_loader, test_loader, class_names = data_setup.create_dataloaders(cfg, M.get_transform(), device)
train_ds, test_ds = train_loader.dataset, test_loader.dataset
counts = pd.DataFrame({"train": data_setup.class_counts(train_ds, class_names),
                       "test": data_setup.class_counts(test_ds, class_names)})
x, y = train_ds[0]
print(f"classes {class_names} | sample tensor {tuple(x.shape)} {x.dtype}, label {class_names[y]!r}")
counts

In [ ]:
# A look at the data: class balance, a few photos, and how much the raw sizes vary
# (which is why every image is resized + center-cropped before it reaches the network).
counts.plot.bar(color=["#2a78d6", "#eb6834"], rot=0, figsize=(6, 3), title="Images per class")
plt.show()

paths = data_setup.image_paths(train_ds)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, path in zip(axes.flat, paths[:: max(1, len(paths) // 8)]):
    ax.imshow(Image.open(path))
    ax.set_title(path.parent.name)
    ax.axis("off")
plt.tight_layout()
plt.show()

pd.DataFrame([Image.open(p).size for p in paths], columns=["width", "height"]).describe().round(0)

## 4. Load model
`model_builder.build_model` creates `model.FoodVisionNet` (from `../model/model.py`) with ImageNet
weights and freezes the backbone, so only `Dropout(0.3) -> Linear(1408, 3)` trains.
`model_builder.load_deployed` loads the checkpoint that is deployed right now (from `../model`, or
from the Hub when the clone has no weights). It is evaluated next to the new run in section 6, and
it stays deployed unless the new run beats it.

In [ ]:
net = model_builder.build_model(cfg, class_names).to(device)
print(net.backbone.classifier)
print(f"parameters: {utils.count_params(net):,} total, {model_builder.trainable_params(net):,} trainable (the head)")

In [ ]:
deployed_net, deployed_dir, deployed_info = model_builder.load_deployed(device)
print("deployed checkpoint:", deployed_info.get("variant", export.DEPLOYED_DEFAULT_NAME) if deployed_net else None)
print("provenance:", deployed_info.get("source"))

## 5. Training
Plain cross-entropy, Adam over the head only, 10 epochs. The whole net stays in `train()` mode while
training (as in the original recipe), so the frozen backbone's BatchNorm statistics adapt to food
photos. The test split is evaluated after every epoch for the curves only: the recipe keeps the last
epoch, there is no model selection on it. A full run took 520 s (~47 s/epoch) on a desktop CPU; on a
GPU it should take about a minute (estimate, not measured).

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = engine.make_optimizer(net, cfg.lr)
history, train_time_s = engine.train(net, train_loader, test_loader, loss_fn, optimizer, cfg.epochs, device)
print(f"training time: {train_time_s:.0f} s")

In [ ]:
curves = export.plot_history(history, ASSETS / "training_curves.png",
                             title=f"Retrain with training/ ({device}): train vs test per epoch")
display(ShowImage(filename=str(curves)))

## 6. Evaluation
Both variants are scored on the same test images: accuracy, macro F1 and per-class accuracy (the
share of each class's photos classified correctly). `export.select_best` keeps the deployed
checkpoint unless the new run is strictly better (accuracy, then macro F1).

In [ ]:
variants = []
if deployed_net is not None:
    result = engine.evaluate(deployed_net, test_loader, loss_fn, device)
    variants.append(export.deployed_variant(deployed_dir, deployed_info, result, class_names))
result = engine.evaluate(net, test_loader, loss_fn, device)
variants.append(export.new_run_variant(net, result, class_names, device, train_time_s))

best = export.select_best(variants)
print("best:", best.name)
pd.DataFrame(export.comparison(variants)).T.round(4)

In [ ]:
cm = export.plot_confusion_matrix(best.y_true, best.y_pred, class_names, ASSETS / "confusion_matrix.png",
                                  title=best.name)
display(ShowImage(filename=str(cm), width=420))
if len(variants) > 1:
    display(ShowImage(filename=str(export.plot_comparison(variants, best, ASSETS / "comparison.png"))))

## 7. Inference
Exactly what the Space and the Inference Endpoint run: the best weights are staged in a folder,
then `model.load(dir, device).predict(image)` returns `{class: probability}` for all three classes.

In [ ]:
staged = export.save_weights(best, cfg.run_dir / "best")
predictor = M.load(staged, device)
examples = sorted((utils.SPACE_DIR / "examples").glob("*.jpg"))
fig, axes = plt.subplots(1, len(examples), figsize=(4 * len(examples), 4))
for ax, path in zip(np.atleast_1d(axes), examples):
    probs = predictor.predict(path)
    top = max(probs, key=probs.get)
    print(path.name, {k: round(v, 4) for k, v in probs.items()})
    ax.imshow(Image.open(path))
    ax.set_title(f"{top} ({probs[top]:.1%})")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Export
Writes the best variant into `cfg.model_dir` (`../model`, or `outputs/smoke/model` for smoke runs):
`model.safetensors` + `config.json`, `metrics.json` with the comparison of every variant, the plots
in `assets/`, and the refreshed model card (metrics + Experiments tables). `model.py`, `handler.py`
and `requirements.txt` are already there. To publish, set `PUSH_TO_HUB = True` in section 2; the
token comes from `hf auth login` (local) or an `HF_TOKEN` secret / environment variable.
Smoke runs are never uploaded.

In [ ]:
metrics = export.build_metrics(
    best, variants,
    data={"n_train": len(train_ds), "n_test": len(test_ds), "n_classes": len(class_names)},
    params=utils.count_params(net), smoke=cfg.smoke,
)
export.export(best, metrics, cfg.model_dir, ASSETS)

for path in sorted(cfg.model_dir.rglob("*")):
    if path.is_file() and not {"__pycache__", ".git"} & set(path.parts):
        print(f"{path.relative_to(cfg.model_dir).as_posix():30s} {path.stat().st_size / 1e6:8.2f} MB")

if PUSH_TO_HUB and not cfg.smoke:
    print(utils.upload_model(cfg.model_dir))